# Project 1: E-Commerce Intelligence — Step 5: Customer Review NLP

> **Branch:** `feature/customer-review-nlp`  
> **Dataset:** Olist Brazilian E-Commerce — `olist_order_reviews_dataset.csv`  
> **Tasks:** Sentiment (positive / neutral / negative) and Urgency (urgent / non-urgent) classification  
> **GPU Stack:** RAPIDS cuML 26.4 + scikit-learn CPU baselines

## Pipeline Overview
| Stage | Description |
|---|---|
| 0 | Setup & Imports (GPU cuML + CPU sklearn + utilities) |
| 1 | Data Ingestion & Label Engineering |
| 2 | Text Pre-processing & Cleaning |
| 3 | Feature Extraction: TF-IDF Vectorisation |
| 4 | Sentiment Classification (GPU LinearSVC + NB baselines) |
| 5 | Urgency Classification (GPU LinearSVC + NB baselines) |
| 6 | HPO — Grid Search on Best GPU Model |
| 7 | Evaluation, Metrics & Visualisations |
| 8 | Model Persistence & Downstream Export |

## Section 0 — Setup & Imports

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import warnings, os, re, joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
PLOT_DIR = 'plots'
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# ── Scikit-Learn: CPU Feature Extraction & NB Baselines ───────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.svm import LinearSVC as skLinearSVC
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, f1_score)
from sklearn.preprocessing import LabelEncoder

# ── RAPIDS cuML: GPU Accelerated Classification ───────────────────────────────
import cuml
from cuml.svm import LinearSVC as cuLinearSVC

cuml.global_settings.output_type = 'numpy'

print('All imports successful ✓')
print(f'RAPIDS cuML version: {cuml.__version__}')

## Section 1 — Data Ingestion & Label Engineering

In [ ]:
# ── Load raw reviews ──────────────────────────────────────────────────────────
REVIEWS_PATH = '../../datasets/ecommerce/olist_order_reviews_dataset.csv'
reviews = pd.read_csv(REVIEWS_PATH)

print(f'Total reviews: {len(reviews):,}')
print(f'Reviews with text: {reviews["review_comment_message"].notna().sum():,}')
print('Score distribution:')
print(reviews['review_score'].value_counts().sort_index())

# ── Merge title + message into single text field ──────────────────────────────
reviews['review_comment_title']   = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
reviews['full_text'] = (reviews['review_comment_title'] + ' ' +
                        reviews['review_comment_message']).str.strip()

# ── Label Engineering ─────────────────────────────────────────────────────────
sentiment_map = {1: 'negative', 2: 'negative', 3: 'neutral',
                 4: 'positive', 5: 'positive'}
reviews['sentiment'] = reviews['review_score'].map(sentiment_map)

reviews['is_urgent'] = (
    (reviews['review_score'] <= 2) &
    (reviews['review_comment_message'].str.len() > 0)
).astype(int)

print('\nSentiment distribution:')
print(reviews['sentiment'].value_counts())
print('\nUrgency distribution:')
print(reviews['is_urgent'].value_counts())

# Keep only rows with actual text for modelling
df_text = reviews[reviews['full_text'].str.len() > 3].copy().reset_index(drop=True)
print(f'\nRows with usable text: {len(df_text):,}')

## Section 2 — Text Pre-processing & Cleaning

In [ ]:
import unicodedata

# Compact Portuguese + English stop-words
PT_STOPWORDS = {
    'de','do','da','dos','das','em','no','na','nos','nas','um','uma','uns','umas',
    'o','a','os','as','e','que','se','para','com','por','mas','ou','ao','aos',
    'pelo','pela','pelos','pelas','eu','tu','ele','ela','eles','elas',
    'me','te','lhe','lhes','meu','minha','teu','tua','seu','sua','isso','este',
    'esta','esse','essa','aquele','aquela','muito','mais','bem','ja','ainda',
    'tambem','quando','como','foi','ser','ter','nao','sim',
    'the','and','of','to','in','is','it','for','on','are','with','this','that',
    'was','at','be'
}

def clean_text(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [t for t in text.split() if t not in PT_STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

df_text['clean_text'] = df_text['full_text'].apply(clean_text)
print('Text cleaning done ✓')
print('Sample cleaned text:')
print(df_text[['full_text','clean_text']].head(3).to_string())

## Section 3 — Feature Extraction: TF-IDF Vectorisation

In [ ]:
tfidf = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=20_000,
    sublinear_tf=True,
    min_df=3
)

X_sparse = tfidf.fit_transform(df_text['clean_text'])
X = X_sparse.toarray().astype(np.float32)   # dense float32 for GPU models
X_bin = (X > 0).astype(np.float32)          # binary for BernoulliNB

print(f'TF-IDF matrix shape : {X.shape}')
print(f'Vocabulary size     : {len(tfidf.vocabulary_):,}')

le_sent = LabelEncoder()
y_sentiment = le_sent.fit_transform(df_text['sentiment'])
y_urgency   = df_text['is_urgent'].values.astype(np.int32)

print(f'Sentiment classes   : {le_sent.classes_}')
print(f'Urgency balance     : {y_urgency.mean():.2%} urgent')

## Section 4 — Sentiment Classification

In [ ]:
print('Training cuML LinearSVC for sentiment (GPU)...')
svc_sent = cuLinearSVC(C=1.0, max_iter=2000)
svc_sent.fit(X, y_sentiment)
y_pred_svc_sent = svc_sent.predict(X).astype(int)
f1_svc_sent = f1_score(y_sentiment, y_pred_svc_sent, average='macro')
print(f'  cuML LinearSVC macro-F1 (train): {f1_svc_sent:.4f}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mnb = MultinomialNB(alpha=0.1)
f1_mnb = cross_val_score(mnb, X_sparse, y_sentiment, cv=cv, scoring='f1_macro').mean()
print(f'  MultinomialNB    CV macro-F1: {f1_mnb:.4f}')

bnb = BernoulliNB(alpha=0.5)
f1_bnb = cross_val_score(bnb, X_bin, y_sentiment, cv=cv, scoring='f1_macro').mean()
print(f'  BernoulliNB      CV macro-F1: {f1_bnb:.4f}')

gnb = GaussianNB()
f1_gnb = cross_val_score(gnb, X, y_sentiment, cv=cv, scoring='f1_macro').mean()
print(f'  GaussianNB       CV macro-F1: {f1_gnb:.4f}')

print('Sentiment classification complete ✓')

## Section 5 — Urgency Classification

In [ ]:
print('Training cuML LinearSVC for urgency (GPU)...')
svc_urg = cuLinearSVC(C=1.0, max_iter=2000)
svc_urg.fit(X, y_urgency)
y_pred_svc_urg = svc_urg.predict(X).astype(int)
f1_svc_urg = f1_score(y_urgency, y_pred_svc_urg, average='binary')
print(f'  cuML LinearSVC binary-F1 (train): {f1_svc_urg:.4f}')

cv_urg = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_mnb_urg = cross_val_score(MultinomialNB(alpha=0.1), X_sparse, y_urgency,
                               cv=cv_urg, scoring='f1').mean()
print(f'  MultinomialNB    CV F1: {f1_mnb_urg:.4f}')

f1_bnb_urg = cross_val_score(BernoulliNB(alpha=0.5), X_bin, y_urgency,
                               cv=cv_urg, scoring='f1').mean()
print(f'  BernoulliNB      CV F1: {f1_bnb_urg:.4f}')

f1_gnb_urg = cross_val_score(GaussianNB(), X, y_urgency,
                               cv=cv_urg, scoring='f1').mean()
print(f'  GaussianNB       CV F1: {f1_gnb_urg:.4f}')

print('Urgency classification complete ✓')

## Section 6 — HPO: Grid Search on LinearSVC

In [ ]:
print('HPO for sentiment LinearSVC...')
param_grid = {'C': [0.01, 0.1, 1.0, 5.0, 10.0]}

gs_sent = GridSearchCV(
    skLinearSVC(max_iter=3000, dual=False),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1
)
gs_sent.fit(X_sparse, y_sentiment)
best_C_sent = gs_sent.best_params_['C']
print(f'  Best C (sentiment): {best_C_sent}  |  CV F1-macro: {gs_sent.best_score_:.4f}')

print('HPO for urgency LinearSVC...')
gs_urg = GridSearchCV(
    skLinearSVC(max_iter=3000, dual=False),
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1
)
gs_urg.fit(X_sparse, y_urgency)
best_C_urg = gs_urg.best_params_['C']
print(f'  Best C (urgency)  : {best_C_urg}  |  CV F1: {gs_urg.best_score_:.4f}')

# Re-train final GPU models with best C
print('Re-training final GPU models with optimal C...')
final_sent_gpu = cuLinearSVC(C=float(best_C_sent), max_iter=2000)
final_sent_gpu.fit(X, y_sentiment)

final_urg_gpu = cuLinearSVC(C=float(best_C_urg), max_iter=2000)
final_urg_gpu.fit(X, y_urgency)

print('HPO & final model training complete ✓')

## Section 7 — Evaluation, Metrics & Visualisations

In [ ]:
y_pred_final_sent = final_sent_gpu.predict(X).astype(int)
print('=== SENTIMENT CLASSIFICATION REPORT (cuML LinearSVC — GPU) ===')
print(classification_report(y_sentiment, y_pred_final_sent,
                             target_names=le_sent.classes_))

fig, ax = plt.subplots(figsize=(7, 5))
cm_sent = confusion_matrix(y_sentiment, y_pred_final_sent)
ConfusionMatrixDisplay(cm_sent, display_labels=le_sent.classes_).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Sentiment — Confusion Matrix (cuML LinearSVC GPU)', pad=12)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/nlp_sentiment_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
y_pred_final_urg = final_urg_gpu.predict(X).astype(int)
print('=== URGENCY CLASSIFICATION REPORT (cuML LinearSVC — GPU) ===')
print(classification_report(y_urgency, y_pred_final_urg,
                             target_names=['non-urgent', 'urgent']))

fig, ax = plt.subplots(figsize=(5, 4))
cm_urg = confusion_matrix(y_urgency, y_pred_final_urg)
ConfusionMatrixDisplay(cm_urg, display_labels=['non-urgent', 'urgent']).plot(
    ax=ax, cmap='Oranges', colorbar=False)
ax.set_title('Urgency — Confusion Matrix (cuML LinearSVC GPU)', pad=12)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/nlp_urgency_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Model Comparison Bar Chart ─────────────────────────────────────────────────
model_names = ['cuML LinearSVC\n(GPU)', 'MultinomialNB\n(CPU)',
               'BernoulliNB\n(CPU)', 'GaussianNB\n(CPU)']
f1_sent_all = [f1_score(y_sentiment, y_pred_final_sent, average='macro'),
               f1_mnb, f1_bnb, f1_gnb]
f1_urg_all  = [f1_score(y_urgency, y_pred_final_urg, average='binary'),
               f1_mnb_urg, f1_bnb_urg, f1_gnb_urg]

x = np.arange(len(model_names))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, f1_sent_all, width, label='Sentiment F1-macro', color='steelblue')
b2 = ax.bar(x + width/2, f1_urg_all,  width, label='Urgency F1',        color='tomato')
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('F1 Score')
ax.set_title('Sentiment vs Urgency — Model Comparison')
ax.legend(); ax.set_ylim(0, 1.1)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/nlp_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Model comparison chart saved.')

In [ ]:
# ── Top TF-IDF Keywords per Sentiment ────────────────────────────────────────
feature_names = np.array(tfidf.get_feature_names_out())
mnb_final = MultinomialNB(alpha=0.1)
mnb_final.fit(X_sparse, y_sentiment)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colours = ['tomato', 'gray', 'steelblue']
for idx, cls in enumerate(le_sent.classes_):
    top_idx = np.argsort(mnb_final.feature_log_prob_[idx])[-15:][::-1]
    top_words  = feature_names[top_idx]
    top_scores = mnb_final.feature_log_prob_[idx][top_idx]
    axes[idx].barh(top_words[::-1], top_scores[::-1], color=colours[idx])
    axes[idx].set_title(f'Top Keywords — {cls.upper()}')
    axes[idx].set_xlabel('Log Probability')
plt.suptitle('Top 15 Keywords per Sentiment Class (MultinomialNB)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/nlp_top_keywords_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Keyword chart saved.')

## Section 8 — Model Persistence & Downstream Export

In [ ]:
joblib.dump(tfidf,          'models/nlp_tfidf_vectoriser.joblib')
joblib.dump(final_sent_gpu, 'models/nlp_sentiment_clf_gpu.joblib')
joblib.dump(final_urg_gpu,  'models/nlp_urgency_clf_gpu.joblib')
joblib.dump(le_sent,        'models/nlp_label_encoder_sentiment.joblib')
print('Models saved ✓')
print('  models/nlp_tfidf_vectoriser.joblib')
print('  models/nlp_sentiment_clf_gpu.joblib')
print('  models/nlp_urgency_clf_gpu.joblib')
print('  models/nlp_label_encoder_sentiment.joblib')

In [ ]:
df_text['pred_sentiment'] = le_sent.inverse_transform(
    final_sent_gpu.predict(X).astype(int))
df_text['pred_is_urgent'] = final_urg_gpu.predict(X).astype(int)

out_cols = ['review_id','order_id','review_score',
            'sentiment','is_urgent','pred_sentiment','pred_is_urgent',
            'full_text','clean_text']
df_text[out_cols].to_csv(
    'outputs/review_predictions.csv.gz', index=False, compression='gzip')

print('Downstream export saved ✓')
print(f'  outputs/review_predictions.csv.gz  ({len(df_text):,} rows)')
print()
print('=== PIPELINE COMPLETE ===')
print(f'Sentiment best CV F1-macro : {gs_sent.best_score_:.4f}  (C={best_C_sent})')
print(f'Urgency   best CV F1       : {gs_urg.best_score_:.4f}  (C={best_C_urg})')